<a href="https://colab.research.google.com/github/CPTR295/Learning-Langchain/blob/main/MultiRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain_openai

In [2]:
import requests
from langchain_core.embeddings import Embeddings
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

class OpenRouterNvidiaEmbeddings(Embeddings):
    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://openrouter.ai/api/v1/embeddings"
        self.model = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

    def _embed(self, texts, input_type):
        response = requests.post(
            self.url,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": self.model,
                "input": [
                    {
                        "content": [
                            {"type": "text", "text": text}
                        ]
                    }
                    for text in texts
                ],
                "input_type": input_type,
                "encoding_format": "float",
            },
            timeout=120,
        )

        response.raise_for_status()
        data = response.json()["data"]
        data.sort(key=lambda item: item["index"])

        return [item["embedding"] for item in data]

    def embed_documents(self, texts):
        return self._embed(texts, "passage")

    def embed_query(self, text):
        return self._embed([text], "query")[0]


embeddings_model = OpenRouterNvidiaEmbeddings(api_key)

In [ ]:
from langchain_core import documents
from typing import Annotated,Literal,TypedDict
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.vectorstores.in_memory import InMemoryVectorStore
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langgraph.graph import END,START,StateGraph
from langgraph.graph.message import add_messages

#embeddings = OpenAIEmbeddings()
embeddings = embeddings_model

model_low_temp =ChatOpenAI( #Many providers provide same api interface as openai including openrouter
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.1
)

model_high_temp =ChatOpenAI( #Many providers provide same api interface as openai including openrouter
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.9
)
class State(TypedDict):
  messages: Annotated[list,add_messages]
  user_query: str
  domain: Literal['records','insurance']
  documents: list[Document]
  answer: str

class Input(TypedDict):
  user_query: str

class Output(TypedDict):
  documents: list[Document]
  answer: str

sample_docs = [
    Document(page_content="Patient medical record...",metadata={'domain': 'records'}),
    Document(page_content="Patient insurance policy...",metadata={'domain': 'insurance'}),
]

#initialize the vector

medical_records_store = InMemoryVectorStore.from_documents(sample_docs,embeddings)
medical_records_retriever = medical_records_store.as_retriever()

insurance_faqs_store = InMemoryVectorStore.from_documents(sample_docs,embeddings)
insurance_faqs_retriever = insurance_faqs_store.as_retriever()

router_prompt = SystemMessage(
    """You need to decide which domain to route the user query to. You have two domains to choose from:
    - records: contains medical records of the patient, such as diagnosis, treatment and prescriptions.
    - insurance: contains frequently asked questions about insurance policies, claims, and coverage.

      Output only the domain name."""
)

def router_node(state: State,)->State:
  user_message = HumanMessage(state['user_query'])
  messages = [router_prompt,*state['messages'],user_message]
  res = model_low_temp.invoke(messages)
  return{
      'domain': res.content,
      'messages': [user_message, res]
  }

def pick_retriever(state: State) -> Literal['retrieve_medical_records','retrieve_insurance_faqs']:
    if state['domain'] == 'records':
      return medical_records_retriever
    else:
      return insurance_faqs_retriever

def retrieve_medical_records(state: State)->State:
    documents = medical_records_retriever.invoke(state['user_query'])
    return {'documents':documents}

def retrieve_insurance_faqs(state: State)->State:
    documents = insurance_faqs_retriever.invoke(state['user_query'])
    return {'documents':documents}

medical_records_prompt = SystemMessage(
    "You are a helpful medical chatbot, who answers questions based on the patient's medical records, such as diagnosis, treatment, and prescriptions."
)

insurance_faqs_prompt = SystemMessage(
    "You are a helpful medical insurance chatbot, who answers frequently asked questions about insurance policies, claims, and coverage."
)

def generate_answer(state: State) -> State:
    if state["domain"] == "records":
        prompt = medical_records_prompt
    else:
        prompt = insurance_faqs_prompt
    messages = [
        prompt,
        *state["messages"],
        HumanMessage(f"Documents: {state['documents']}"),
    ]
    res = model_high_temp.invoke(messages)
    return {
        "answer": res.content,
        # update conversation history
        "messages": res,
    }


builder = StateGraph(State, input=Input, output=Output)
builder.add_node("router", router_node)
builder.add_node("retrieve_medical_records", retrieve_medical_records)
builder.add_node("retrieve_insurance_faqs", retrieve_insurance_faqs)
builder.add_node("generate_answer", generate_answer)
builder.add_edge(START, "router")
builder.add_conditional_edges("router", pick_retriever)
builder.add_edge("retrieve_medical_records", "generate_answer")
builder.add_edge("retrieve_insurance_faqs", "generate_answer")
builder.add_edge("generate_answer", END)

graph = builder.compile()

# Example usage
input = {"user_query": "Am I covered for COVID-19 treatment?"}
for chunk in graph.stream(input):
    print(chunk)